In [225]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [226]:
%pip install sqlalchemy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [227]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [228]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta

from dotenv import load_dotenv

load_dotenv()

True

In [229]:
os.getenv("REGION")

'ap-south-1'

### ingest_daily_support_tickets

In [230]:
# ---------- CONFIG ----------
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

S3_BUCKET = "awscareplusdata" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [231]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [232]:
# ---------- UTILITY FUNCTIONS ----------
def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Uploaded to s3://{bucket}/{key}")

def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head())

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()

(24, 10)
    ticket_id        created_at       resolved_at  agent priority  \
0  TCK0731000  2025-07-31 00:39  2025-08-01 11:16  Kavya    Medum   
1  TCK0731001  2025-07-31 00:58  2025-07-31 14:25  Rohit   Medium   
2  TCK0731002  2025-07-31 01:47  2025-07-31 11:47  Sneha   Medium   
3  TCK0731002  2025-07-31 01:47  2025-07-31 11:47  Sneha   Medium   
4  TCK0731003  2025-07-31 01:53  2025-08-01 07:09  Sneha      Low   

  num_interactions         IssUeCat   channel    status agent_feedback  
0                5  Feature Request     Email  Resolved                 
1                4      Login Issue     Email  Resolved                 
2          -999999   Account Locked  Web Form  Resolved                 
3          -999999   Account Locked  Web Form  Resolved                 
4                8   Account Locked      Chat  Resolved                 
✅ Uploaded to s3://awscareplusdata/support-tickets/raw/support_tickets_2025-07-31.csv
📅 Updated tracker to 2025-07-31


In [233]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

S3_BUCKET = "awscareplusdata"
S3_PREFIX = "support-tickets/raw/"

AWS_CONFIG = {
    'aws_access_key_id': os.getenv("AWS_ACCESS_KEY"),
    'aws_secret_access_key': os.getenv("SECRET_KEY"),
    'region_name': os.getenv("REGION")
}

print("Region:", AWS_CONFIG["region_name"])
print("Bucket:", S3_BUCKET)

Region: ap-south-1
Bucket: awscareplusdata


In [235]:
import boto3
import time

s3 = boto3.client('s3', **AWS_CONFIG)

# List all support ticket CSV files in the raw folder
response = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_PREFIX
)

files = [
    obj['Key']
    for obj in response.get('Contents', [])
    if obj['Key'].endswith('.csv')
]

files.sort()

print(f"Found {len(files)} raw CSV files:")
for key in files:
    print(key)

# Re-upload each file to trigger the updated Lambda
for key in files:
    response = s3.get_object(
        Bucket=S3_BUCKET,
        Key=key
    )

    file_data = response['Body'].read()

    s3.put_object(
        Bucket=S3_BUCKET,
        Key=key,
        Body=file_data
    )

    print(f"✅ Re-uploaded: {key}")
    time.sleep(1)

print("\n🎉 All raw ticket files re-uploaded.")
print("The updated Lambda will regenerate the processed Parquet files.")

Found 31 raw CSV files:
support-tickets/raw/support_tickets_2025-07-01.csv
support-tickets/raw/support_tickets_2025-07-02.csv
support-tickets/raw/support_tickets_2025-07-03.csv
support-tickets/raw/support_tickets_2025-07-04.csv
support-tickets/raw/support_tickets_2025-07-05.csv
support-tickets/raw/support_tickets_2025-07-06.csv
support-tickets/raw/support_tickets_2025-07-07.csv
support-tickets/raw/support_tickets_2025-07-08.csv
support-tickets/raw/support_tickets_2025-07-09.csv
support-tickets/raw/support_tickets_2025-07-10.csv
support-tickets/raw/support_tickets_2025-07-11.csv
support-tickets/raw/support_tickets_2025-07-12.csv
support-tickets/raw/support_tickets_2025-07-13.csv
support-tickets/raw/support_tickets_2025-07-14.csv
support-tickets/raw/support_tickets_2025-07-15.csv
support-tickets/raw/support_tickets_2025-07-16.csv
support-tickets/raw/support_tickets_2025-07-17.csv
support-tickets/raw/support_tickets_2025-07-18.csv
support-tickets/raw/support_tickets_2025-07-19.csv
support